# Klasifikasi Kategori dan Prioritas Keluhan Mahasiswa Berbasis NLP

Notebook ini berisi alur Machine Learning dari dataset dummy sampai evaluasi model.

In [1]:
import pandas as pd
import re, string
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
import joblib


KeyboardInterrupt: 

In [ ]:
df = pd.read_csv('data_keluhan_dummy.csv')
df.head()


In [ ]:
df['kategori'].value_counts(), df['prioritas'].value_counts()


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['keluhan_bersih'] = df['keluhan'].apply(clean_text)
df[['keluhan', 'keluhan_bersih']].head()


In [ ]:
def train_and_evaluate(target_col):
    X = df['keluhan_bersih']
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    models = {
        'Naive Bayes': MultinomialNB(),
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Linear SVM': LinearSVC(),
        'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42)
    }

    results = []
    best_model = None
    best_score = -1
    best_name = ''

    for name, clf in models.items():
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
            ('model', clf)
        ])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        acc = accuracy_score(y_test, pred)
        results.append([name, acc])
        if acc > best_score:
            best_score = acc
            best_model = pipe
            best_name = name
            best_pred = pred

    print('Target:', target_col)
    print('Model terbaik:', best_name)
    print('Akurasi:', best_score)
    print('\nClassification Report:')
    print(classification_report(y_test, best_pred))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, best_pred))
    return pd.DataFrame(results, columns=['Algoritma', 'Akurasi']), best_model

hasil_kategori, model_kategori = train_and_evaluate('kategori')
hasil_kategori


In [ ]:
hasil_prioritas, model_prioritas = train_and_evaluate('prioritas')
hasil_prioritas


In [ ]:
contoh_keluhan = ['WiFi di gedung B sering mati saat kelas online']
print('Kategori:', model_kategori.predict(contoh_keluhan)[0])
print('Prioritas:', model_prioritas.predict(contoh_keluhan)[0])
